In [2]:
import ee
import os
import pandas as p
import geopandas as gpd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np


In [3]:
PROJECT_ROOT = Path().resolve().parent

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

True

In [3]:
ee.Authenticate()



Successfully saved authorization token.


In [4]:
ee.Initialize(project=os.environ["EE_PROJECT"])

In [73]:

sys.path.append(str(PROJECT_ROOT / "src"))

from gee_utils import create_images_for_all_locations, get_samples, export_patches
from data_utils import make_padded_bbox_all_location, clean_labelled_data


In [72]:
import importlib
import data_utils
import gee_utils

importlib.reload(data_utils)
importlib.reload(gee_utils)



<module 'gee_utils' from '/Users/bensutton/Projects/dissertation-code/src/gee_utils.py'>

In [6]:
# Want to upload the sites.json to ee and then create image collection for each, clip to the bbox, need to select the dates for each location
# Could save the location in the json or just use the date from the points? 


site_fp = PROJECT_ROOT / "configs" / "sites.json"
label_fp = PROJECT_ROOT / "configs" / "labels.gpkg"
cleaned_label_fp = PROJECT_ROOT / "configs" / "cleaned_labels.gpkg"

In [36]:
clean_labelled_data(label_fp= label_fp, cleaned_label_fp = cleaned_label_fp)

Saved a cleaned version of /Users/bensutton/Projects/dissertation-code/configs/labels.gpkg as /Users/bensutton/Projects/dissertation-code/configs/cleaned_labels.gpkg


In [43]:
# test the created cleaned labels geopackage


labels_gdf = gpd.read_file(cleaned_label_fp)

labels_gdf.head()


,label_id,longitude,latitude,location,obs_date,comparison_dates_used,s2_target_image_id,s2_old_image_id,s1_image_id,class_label,notes,created_at,updated_at,created_by,class_int,lc,geometry
0,HA0001_20210121,27.823149,-25.752589,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:04.027316+00:00,2026-05-14T14:19:04.027316+00:00,bensutton,3,1,POINT (27.82315 -25.75259)
1,HA0002_20210121,27.807339,-25.760474,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:06.695977+00:00,2026-05-14T14:19:06.695977+00:00,bensutton,3,1,POINT (27.80734 -25.76047)
2,HA0003_20210121,27.809334,-25.755411,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:10.017332+00:00,2026-05-14T14:19:10.017332+00:00,bensutton,3,1,POINT (27.80933 -25.75541)
3,HA0004_20210121,27.805043,-25.758155,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:13.638342+00:00,2026-05-14T14:19:13.638342+00:00,bensutton,3,1,POINT (27.80504 -25.75816)
4,HA0005_20210121,27.819760,-25.762774,Hartbeespoort,2021-01-21,"[""2020-12-30"", ""2020-09-23""]",COPERNICUS/S2_SR_HARMONIZED/20210121T075231_20...,COPERNICUS/S2_SR_HARMONIZED/20201230T080239_20...,COPERNICUS/S1_GRD/S1A_IW_GRDH_1SDV_20210121T16...,floating_plants,,2026-05-14T14:19:18.937682+00:00,2026-05-14T14:19:18.937682+00:00,bensutton,3,1,POINT (27.81976 -25.76277)


In [40]:
# Make a bounding box for each site based on the total bounds of the labelled points and add a padding

make_padded_bbox_all_location(sites_file= site_fp, project_root= PROJECT_ROOT)

Created padded bbox from the points files for Vembanad, Winam, Inle, Hartbeespoort, Mula, RawaPening, Rodman, Valsequillo


In [69]:
image_collection = create_images_for_all_locations(sites_file= site_fp, cleaned_label_fp=cleaned_label_fp, clip = False)

Created image collection for: Vembanad
Created image collection for: Winam
Created image collection for: Inle
Created image collection for: Hartbeespoort
Created image collection for: Mula
Created image collection for: RawaPening
Created image collection for: Rodman
Created image collection for: Valsequillo


In [74]:
# Due to memory limits it is not possible to convert all the sampled points to a data frame, therefore the sampled points for each location 
# is export to drive as a CSV separately


# Trying to save all without aweip95

get_samples(merged_ic=image_collection,cleaned_label_fp= cleaned_label_fp)

Exporting sampled points for Hartbeespoort to drive/Dissertation
Exporting sampled points for Mula to drive/Dissertation
Exporting sampled points for RawaPening to drive/Dissertation
Exporting sampled points for Vembanad to drive/Dissertation
Exporting sampled points for Valsequillo to drive/Dissertation
Exporting sampled points for Rodman to drive/Dissertation
Exporting sampled points for Winam to drive/Dissertation
Exporting sampled points for Inle to drive/Dissertation


In [56]:
for task in ee.batch.Task.list():
    status = task.status()
    desc = status.get("description", "")

    if desc.startswith("sampled_points_"):
        print(desc, status["state"])
        print("EECU seconds:", status.get("batch_eecu_usage_seconds"))
        if status["state"] == "FAILED":
            print(status.get("error_message"))

sampled_points_Inle READY
EECU seconds: None
sampled_points_Winam READY
EECU seconds: None
sampled_points_Rodman READY
EECU seconds: None
sampled_points_Valsequillo READY
EECU seconds: None
sampled_points_Vembanad READY
EECU seconds: None
sampled_points_RawaPening READY
EECU seconds: None
sampled_points_Mula READY
EECU seconds: None
sampled_points_Hartbeespoort RUNNING
EECU seconds: 6613.064941406
sampled_points_Inle CANCELLED
EECU seconds: None
sampled_points_Winam CANCELLED
EECU seconds: None
sampled_points_Rodman CANCELLED
EECU seconds: None
sampled_points_Valsequillo CANCELLED
EECU seconds: None
sampled_points_Vembanad CANCELLED
EECU seconds: None
sampled_points_RawaPening CANCELLED
EECU seconds: None
sampled_points_Mula CANCELLED
EECU seconds: None
sampled_points_Hartbeespoort CANCELLED
EECU seconds: 15775.887695312
sampled_points_Rodman COMPLETED
EECU seconds: 11354.2392578125
sampled_points_Rodman CANCELLED
EECU seconds: 8802.4375
sampled_points_Vembanad COMPLETED
EECU seconds: 

In [ ]:
# Load location samples from csvs in outputs/sampled_points_by_location, combine into a single dataframe and save 

samples_outpath = PROJECT_ROOT / "outputs" / "sample_points.csv"

with open(site_fp) as f:
    sites= json.load(f)

location_list = list(sites.get("sites", {}).keys())

list_of_location_dfs = []

for location in location_list:
    location_samples_fp = PROJECT_ROOT / "outputs/sampled_points_by_location" / f"..." 

    location_df = pd.read_csv(location_samples_fp)

    list_of_location_dfs.append(location_df)

all_samples = pd.concat(list_of_location_dfs)

all_samples.to_csv(samples_outpath)

In [49]:
len(samples.loc[samples["binary"] == 1.0])

461

In [51]:
samples.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 116 entries, 0 to 115
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   B11       116 non-null    int64  
 1   B12       116 non-null    int64  
 2   B2        116 non-null    int64  
 3   B3        116 non-null    int64  
 4   B4        116 non-null    int64  
 5   B5        116 non-null    int64  
 6   B6        116 non-null    int64  
 7   B7        116 non-null    int64  
 8   B8        116 non-null    int64  
 9   B8A       116 non-null    int64  
 10  lat       116 non-null    float64
 11  lc        114 non-null    float64
 12  location  116 non-null    object 
 13  lon       116 non-null    float64
 14  obs_date  116 non-null    object 
dtypes: float64(3), int64(10), object(2)
memory usage: 13.7+ KB


In [ ]:
mula_wh_samples = samples.loc[(samples["location"]=="mula") & (samples["lc"]== 1.0), ["B11", "B12", "B3", "B4", "B5", "B8"]]
mula_non_wh_samples = samples.loc[(samples["location"]=="mula")& (samples["lc"]==0.0), ["B11", "B12", "B3", "B4", "B5", "B8"]]
bins = np.linspace(0,8000, 20)

for col in ["B11", "B12", "B3", "B4", "B5", "B8"]:
    plt.hist(mula_wh_samples[col], bins, alpha=0.5, label='x')
    plt.hist(mula_non_wh_samples[col], bins, alpha=0.5, label='y')
    plt.legend(loc='upper right')
    plt.title(f"{col}")
    plt.show()



In [ ]:
export_patches(merged_ic = image_collection, cleaned_label_fp= cleaned_label_fp, kernel_size= 15)

Exporting sampled patches to drive/Dissertation


In [11]:
import importlib

importlib.reload(gee_utils)
importlib.reload(data_utils)


NameError: name 'gee_utils' is not defined